#**FLASK-NGROK**

A Flask-based backend API was developed to enable interaction with the fine-tuned TinyLlama TinyLlama Educational AI Assistant model. The Flask framework was used because of its lightweight architecture, simplicity, and suitability for deploying machine learning models as web services.

Several required libraries were installed, including:

Flask
Transformers
Pyngrok
Streamlit
PyTorch

The fine-tuned TinyLlama model and tokenizer were loaded directly from the Hugging Face Hub. A text-generation pipeline was then created using the Hugging Face Transformers library to generate responses for educational and programming-related user queries.

The Flask application provided two main API routes:

a home route to verify that the API server was running successfully
a /generate endpoint that accepted user prompts and returned generated responses in JSON format

The input prompts were formatted according to the TinyLlama conversational chat template before being passed to the model. Text generation parameters such as temperature, top-p sampling, and repetition penalty were configured to improve response quality and conversational coherence.

To make the locally hosted Flask server publicly accessible, ngrok was integrated into the application. ngrok generated a secure public URL that exposed the Flask API over the internet, allowing external applications and frontend interfaces to interact with the Educational AI Assistant model in real time.

Key Features of the Flask Backend
Loaded the fine-tuned TinyLlama Educational AI Assistant model from the Hugging Face Hub
Developed REST API endpoints using Flask
Implemented text generation using the Hugging Face Transformers pipeline
Formatted prompts using the TinyLlama conversational template
Returned generated responses in JSON format
Integrated ngrok for public API accessibility
Enabled real-time communication between frontend applications and the trained model

In [1]:
!pip install -q flask
!pip install -q pyngrok
!pip install -q transformers
!pip install -q accelerate
!pip install -q streamlit

In [2]:
!pkill -f flask_app.py

!pkill -f streamlit

!pkill -f ngrok

In [3]:
%%writefile flask_app.py

import torch

from flask import Flask, request, jsonify

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline
)

# =====================================================
# CREATE FLASK APP
# =====================================================

app = Flask(__name__)

# =====================================================
# LOAD MODEL
# =====================================================

model_name = "Srr1234/EduGPT-TinyLlama"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100
)

print("MODEL LOADED SUCCESSFULLY!")

# =====================================================
# TEXT GENERATION FUNCTION
# =====================================================

def generate_text(prompt):

    formatted_prompt = f'''
<|system|>
You are an educational AI assistant.

<|user|>
{prompt}

<|assistant|>
'''

    result = pipe(
        formatted_prompt,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    return result[0]["generated_text"]

# =====================================================
# HOME ROUTE
# =====================================================

@app.route("/", methods=["GET"])

def home():

    return jsonify({
        "message": "EduGPT API Running Successfully"
    })

# =====================================================
# GENERATE ROUTE
# =====================================================

@app.route("/generate", methods=["POST"])

def generate():

    data = request.get_json()

    prompt = data["prompt"]

    answer = generate_text(prompt)

    return jsonify({
        "generated_text": answer
    })

# =====================================================
# RUN APP
# =====================================================

app.run(
    host="0.0.0.0",
    port=5000
)

Overwriting flask_app.py


In [4]:
!python flask_app.py > flask_log.txt 2>&1 &

In [5]:
from pyngrok import ngrok

ngrok.set_auth_token(
    "3DVALNsdmhLDWOFnzUDXRHUYIsi_5J6Vb6iy2eLK8V39PhE9M"
)

flask_tunnel = ngrok.connect(5000)

FLASK_URL = flask_tunnel.public_url

print("FLASK URL:\n")

print(FLASK_URL)

print("\nGENERATE ENDPOINT:\n")

print(f"{FLASK_URL}/generate")

FLASK URL:

https://imagines-levitate-scenic.ngrok-free.dev

GENERATE ENDPOINT:

https://imagines-levitate-scenic.ngrok-free.dev/generate


In [7]:
import requests

url = "https://imagines-levitate-scenic.ngrok-free.dev/generate"

data = {
    "prompt": "what is AI in education?"
}

response = requests.post(
    url,
    json=data
)

print(response.json())

{'generated_text': '\n<|system|>\nYou are an educational AI assistant.\n\n<|user|>\nwhat is AI in education?\n\n<|assistant|>\nAI in education is the use of advanced technology and machine learning to improve the learning experience for students and teachers. It involves the integration of computer programming, artificial intelligence, and other technologies to provide personalized learning experiences, improve student engagement, and enhance the overall effectiveness of teaching and learning. AI in education can be used in a variety of ways, from helping students learn new skills to providing personalized feedback on their progress. It can also be used to automate some of the'}


#**STREAMLIT**

In [8]:
%%writefile app.py

import streamlit as st
import requests

# =====================================================
# PAGE SETTINGS
# =====================================================

st.set_page_config(
    page_title="EduGPT Chatbot",
    page_icon="🎓",
    layout="centered"
)

# =====================================================
# TITLE
# =====================================================

st.title("🎓 EduGPT Educational AI Assistant")

st.write(
    "Ask programming, AI, database, or computer science questions."
)

# =====================================================
# FLASK API URL
# =====================================================

API_URL = "https://imagines-levitate-scenic.ngrok-free.dev/generate"

# =====================================================
# SESSION STATE
# =====================================================

if "messages" not in st.session_state:

    st.session_state.messages = []

# =====================================================
# DISPLAY CHAT HISTORY
# =====================================================

for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])

# =====================================================
# USER INPUT
# =====================================================

prompt = st.chat_input(
    "Ask your educational question..."
)

# =====================================================
# PROCESS QUESTION
# =====================================================

if prompt:

    st.chat_message("user").markdown(prompt)

    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    try:

        response = requests.post(
            API_URL,
            json={
                "prompt": prompt
            },
            timeout=120
        )

        result = response.json()

        answer = result["generated_text"]

    except Exception as e:

        answer = f"ERROR: {str(e)}"

    with st.chat_message("assistant"):

        st.markdown(answer)

    st.session_state.messages.append({
        "role": "assistant",
        "content": answer
    })

Overwriting app.py


In [9]:
!streamlit run app.py --server.port 8501 > streamlit_log.txt 2>&1 &

In [10]:
streamlit_tunnel = ngrok.connect(8501)

STREAMLIT_URL = streamlit_tunnel.public_url

print("STREAMLIT URL:\n")

print(STREAMLIT_URL)

STREAMLIT URL:

https://imagines-levitate-scenic.ngrok-free.dev


In [11]:
!cat streamlit_log.txt

In [12]:
!cat flask_log.txt

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3567.78it/s, Materializing param=model.norm.weight]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
MODEL LOADED SUCCESSFULLY!
 * Serving Flask app 'flask_app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit
127.0.0.1 - - [13/May/2026 08:49:51] "GET /_stcore/host-config HTTP/1.1" 404 -
127.0.0.1 - - [13/May/2026 08:49:51] "GET /_stcore/health HTTP/1.1" 404 -
127.0.0.1 - - [13/May/2026 08:49:54] "GET /_stcore/host-config HTTP/1.1" 404 -
127.0.0.1 - - [13/May/2026 08:49:54] "GET /_stcore/health HTTP/1.1" 404 -
127.0.0.1 - - [13/May/2026 08:49:57] "GET /_stcore/health HTTP/1.1" 404 -
127.0.0.1 - - [13/May/2026 08:49:57] "GET /_stcore/

#**Project Summary**

Installed required libraries such as Flask, Streamlit, Transformers, Accelerate, and Pyngrok to build and deploy the EduGPT chatbot system.

Developed a Flask backend API to host the fine-tuned TinyLlama EduGPT model for text generation.

Loaded the fine-tuned TinyLlama model and tokenizer from Hugging Face
 for inference.

Created REST API endpoints using Flask:

/ route for checking API status

/generate route for generating chatbot responses using POST requests.

Integrated the Hugging Face Transformers pipeline for educational question-answer generation.

Used Ngrok to expose the local Flask server publicly over the internet.
Generated a secure public API endpoint using Ngrok so the chatbot backend could be accessed externally.

Tested the Flask API using Python requests.post() calls to verify successful response generation.

Developed a Streamlit-based chatbot user interface for interactive conversations.

Integrated the Streamlit frontend with the Flask backend API using HTTP POST requests.

Implemented a chat-style UI using Streamlit’s st.chat_input() and st.chat_message() components.

Added session state management in Streamlit to preserve chatbot conversation history.

Connected the Streamlit application to the Flask /generate endpoint for real-time AI-generated responses.

Exposed the Streamlit application publicly using a separate Ngrok tunnel.
Successfully deployed the complete end-to-end EduGPT Educational AI Assistant consisting of:

TinyLlama fine-tuned model

Flask backend API

Streamlit chatbot frontend

Ngrok public deployment.